# ComfyUI on Paperspace

This notebook is adapted from the Kaggle ComfyUI notebook to work on Paperspace Gradient.

## Session options:
 * Machine: Free-P5000, A4000, or any GPU instance
 * Container: Default (Python 3.10+)
 * Auto-shutdown: Recommended

## Directory Structure:
 * `/notebooks` - Persistent working directory
 * `/storage` - Persistent shared storage (great for models)
 * `/tmp` - Temporary storage (cleared on restart)


# Installation

Session options:
 * Machine: Free-P5000 or better GPU (A4000, A6000, etc.)
 * Container: Use default container with Python 3.10+
 * Workspace: `/notebooks` is persistent storage


In [ ]:
%%time
# --- Configuration --- #
# Set to True to update ComfyUI and ComfyUI-Manager to latest versions
update_comfyui = True
update_manager = True

# Set to True to reinstall PyTorch (useful if you have CUDA errors)
reinstall_pytorch = False
# ---------------------- #

import os
import stat
import subprocess

# Paperspace paths
# /notebooks - persistent working directory
# /storage - persistent shared storage (great for models)
# /tmp - temporary storage (cleared on restart)

home_dir = '/notebooks'
venv_dir = f'{home_dir}/venv'
python = f'{venv_dir}/bin/python'
pip = f'{venv_dir}/bin/pip'

# Use /storage for persistent model storage across sessions
model_storage = '/storage/comfyui-models'

def find_bin_folders(folder_path):
    """Find all bin folders and set execute permissions."""
    bin_folders = []
    for root, dirs, files in os.walk(folder_path):
        for dir_name in dirs:
            if dir_name == 'bin':
                bin_folders.append(os.path.join(root, dir_name)) 
    return bin_folders

def fix_venv_permissions():
    """Fix execute permissions on venv bin files."""
    bin_folders = find_bin_folders(venv_dir)
    if bin_folders:
        print("Fixing venv permissions...")
        for bin_folder in bin_folders:
            for filename in os.listdir(bin_folder):
                file_path = os.path.join(bin_folder, filename)
                if os.path.isfile(file_path):
                    current_permissions = os.stat(file_path).st_mode
                    os.chmod(file_path, current_permissions | stat.S_IXUSR | stat.S_IXGRP | stat.S_IXOTH)

def install_pytorch():
    """Install PyTorch with CUDA 12.1 support."""
    print("Installing PyTorch with CUDA 12.1 support...")
    get_ipython().system(f'{pip} install --upgrade pip')
    get_ipython().system(f'{pip} install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121')

def verify_pytorch():
    """Verify PyTorch installation and CUDA support."""
    print("\nVerifying PyTorch installation...")
    result = subprocess.run([python, '-c', 
        'import torch; print(f"PyTorch: {torch.__version__}"); print(f"CUDA available: {torch.cuda.is_available()}"); print(f"CUDA version: {torch.version.cuda}" if torch.cuda.is_available() else "")'],
        capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print(f"Error: {result.stderr}")
        return False
    return "CUDA available: True" in result.stdout

# Install virtualenv if needed
!pip install virtualenv

# Create venv if it doesn't exist
if not os.path.exists(venv_dir):
    print('Creating virtual environment...')
    os.chdir(home_dir)
    get_ipython().system('virtualenv venv -p $(which python3.10)')
    install_pytorch()
else:
    print("Virtual environment exists.")
    fix_venv_permissions()
    
    # Reinstall PyTorch if requested or if verification fails
    if reinstall_pytorch:
        print("Reinstalling PyTorch as requested...")
        get_ipython().system(f'{pip} uninstall -y torch torchvision torchaudio')
        install_pytorch()

# Ensure python symlinks exist
if not os.path.exists(f'{venv_dir}/bin/python3.10'):
    get_ipython().system(f'cp /usr/bin/python3.10 {venv_dir}/bin/')
if not os.path.exists(f'{venv_dir}/bin/python'):
    get_ipython().system(f'ln -s {venv_dir}/bin/python3.10 {venv_dir}/bin/python')
if not os.path.exists(f'{venv_dir}/bin/python3'):
    get_ipython().system(f'ln -s {venv_dir}/bin/python3.10 {venv_dir}/bin/python3')

# Verify PyTorch installation
if not verify_pytorch():
    print("\n⚠️  PyTorch CUDA not working. Reinstalling...")
    get_ipython().system(f'{pip} uninstall -y torch torchvision torchaudio')
    install_pytorch()
    if not verify_pytorch():
        print("\n❌ PyTorch installation failed. Please check your GPU and CUDA drivers.")

# Clone/Update ComfyUI
%cd /notebooks
if not os.path.exists('/notebooks/ComfyUI'):
    !git clone https://github.com/comfyanonymous/ComfyUI.git
%cd ComfyUI
!git fetch origin

if update_comfyui:
    print("\nUpdating ComfyUI to latest version...")
    !git checkout master
    !git pull origin master
else:
    # Pin to stable release v0.11.1
    !git checkout b0d9708974f50fce7d2448ac84e9260c87f7ade3

# Install ComfyUI requirements
print("\nInstalling ComfyUI requirements...")
!{pip} install -r requirements.txt

# Create persistent model storage in /storage
print("\nSetting up model storage...")
!mkdir -p {model_storage}/checkpoints
!mkdir -p {model_storage}/clip
!mkdir -p {model_storage}/vae
!mkdir -p {model_storage}/unet
!mkdir -p {model_storage}/loras

# Link model directories to persistent storage
model_dirs = ['checkpoints', 'clip', 'vae', 'unet', 'loras']
for model_dir in model_dirs:
    src = f'{model_storage}/{model_dir}'
    dst = f'/notebooks/ComfyUI/models/{model_dir}'
    if os.path.islink(dst):
        os.unlink(dst)
    elif os.path.isdir(dst):
        get_ipython().system(f'rm -rf {dst}')
    get_ipython().system(f'ln -s {src} {dst}')

# Create temp model directory
temp_models = '/tmp/comfyui-temp/temp-models'
!mkdir -p /tmp/comfyui-temp
!mkdir -p {temp_models}

checkpoints = '/notebooks/ComfyUI/models/checkpoints'
link_path = checkpoints + '/temp-models'
if not os.path.exists(link_path):
    get_ipython().system(f'ln -s {temp_models} {checkpoints}')

# Install the node manager
print("\nInstalling ComfyUI-Manager...")
%cd /notebooks/ComfyUI/custom_nodes
if not os.path.exists('/notebooks/ComfyUI/custom_nodes/ComfyUI-Manager'):
    !git clone https://github.com/ltdrdata/ComfyUI-Manager.git
%cd ComfyUI-Manager
if update_manager:
    print("Updating ComfyUI-Manager...")
    !git pull

# Pinggy script for tunneling
!wget -q https://raw.githubusercontent.com/wandaweb/jupyter-webui-tunneling/main/pinggy.py -O /notebooks/pinggy.py

# Final verification
print("\n" + "="*50)
verify_pytorch()
print("="*50)
print("\n✅ Installation complete!")
print("📁 Models will be stored persistently in /storage/comfyui-models")
print("\n🚀 Run the next cell to start ComfyUI with Pinggy or Zrok tunneling")


--- 
# WebUI

## Start the WebUI with Pinggy
* Wait for the GUI to start.  
* Click the link that ends with .pinggy.link 😁
* If generation is still running after the link expires in an hour, wait for the generation to complete and restart the WebUI code block to get a new link

In [ ]:
# Starting the Web UI with pinggy

%cd /notebooks/ComfyUI
!python /notebooks/pinggy.py --command='/notebooks/venv/bin/python /notebooks/ComfyUI/main.py' --port=8188

## Start the WebUI with Zrok

### Install Zrok

In [ ]:
# Install Zrok (only needs to run once)

!mkdir /notebooks/zrok
%cd /notebooks/zrok
!rm zrok*.gz
!wget https://github.com/openziti/zrok/releases/download/v1.0.4/zrok_1.0.4_linux_amd64.tar.gz
!tar -xvf ./zrok*.gz 
!chmod a+x /notebooks/zrok/zrok 

### Create a Zrok account
Enter your email address in the email variable

In [ ]:
email = '####@gmail.com' # replace with your email

# --------------

cmd = '/notebooks/zrok/zrok invite'
log = '/notebooks/zrok/log.txt'

!pip install pexpect
!touch $log

import pexpect
import time
child = pexpect.spawn('bash')
child.sendline(f'{cmd} | tee {log}')
child.expect('enter and confirm your email address...')
time.sleep(1); child.sendline(email); time.sleep(1); child.send(chr(9)); time.sleep(1)
child.sendline(email); time.sleep(1); child.send('\n'); time.sleep(1); child.send(chr(9))
time.sleep(1); child.send('\r\n'); time.sleep(2); child.close()
!cat $log
!rm $log

### Enable Zrok 
Paste your Zrok token in the token variable

In [ ]:
# Enable Zrok (needs to run once per instance)
# Paste your Zrok token in the token variable

token = ""
!chmod a+x /notebooks/zrok/zrok 
!/notebooks/zrok/zrok enable $token

### Start the WebUI with Zrok

In [ ]:
# Start the WebUI with Zrok
%cd /notebooks/ComfyUI
command = '/notebooks/venv/bin/python /notebooks/ComfyUI/main.py'
port = '8188'
# ------------------------

!chmod a+x /notebooks/zrok/zrok 
cmd = f'{command} & /notebooks/zrok/zrok share public http://localhost:{port} --headless'
get_ipython().system(cmd)

---
# Model Management

## Install a model

Copy the model URL to the model_url field. Make sure the model can be accessed publicly, without being signed into a website.

In [ ]:
#### Install a model in permanent storage
# Make sure Persistence is set to "Files only" or "Variables and Files"
model_url = 'https://civitai.com/api/download/models/782002'
model_name = 'JuggernautXL.safetensors'

%cd $checkpoints
get_ipython().system(f'wget -O "{model_name}" "{model_url}"')

In [ ]:
# Install a LoRA in permanent storage
model_url = 'https://civitai.com/api/download/models/137124?type=Model&format=SafeTensor'
model_name = 'DreamArt.safetensors'

%cd /notebooks/ComfyUI/models/loras
get_ipython().system(f'wget -O "{model_name}" "{model_url}"')

In [ ]:
# Install a model in temporary storage
#model_url = 'https://civitai.com/api/download/models/160191?type=Model&format=SafeTensor&size=full&fp=fp16'
#model_name = 'YamersRealism.safetensors'
model_url = 'https://civitai.com/api/download/models/456751'
model_name = 'HelloWorld-XL.safetensors' 

%cd $temp_models
get_ipython().system(f'wget -O "{model_name}" "{model_url}"')

## Download a model for a custom node

In [ ]:
model_folder = '/notebooks/ComfyUI/custom_nodes/my_node/models'
model_url = ''
model_name = 'model.safetensors'

%cd $model_folder
get_ipython().system(f'wget -O "{model_name}" "{model_url}"')

---
# File Browser

## Install FileBrowser

In [ ]:
%cd /notebooks
!wget https://github.com/filebrowser/filebrowser/releases/download/v2.27.0/linux-amd64-filebrowser.tar.gz
!tar xvfz linux-amd64-filebrowser.tar.gz
!chmod a+x /notebooks/filebrowser
!/notebooks/filebrowser config init 
!/notebooks/filebrowser config set --auth.method=noauth > /dev/null
!/notebooks/filebrowser config set --branding.theme=dark > /dev/null
!/notebooks/filebrowser users add admin admin 
!/notebooks/filebrowser config export "/notebooks/config.json"

## Run FileBrowser

In [ ]:
%cd /notebooks
!chmod a+x /notebooks/filebrowser

!python /notebooks/pinggy.py --command='/notebooks/filebrowser -c "/notebooks/config.json"' --port=8080

# 
# Delete a model

In [ ]:
# List permanent models
!ls -la $checkpoints

# Delete a model
model_to_delete = '/notebooks/ComfyUI/models/checkpoints/model.safetensors'
!rm $model_to_delete

In [ ]:
# Check the size of a model
!du -sh /notebooks/ComfyUI/models/loras/harrlogos.safetensors

# 
# Delete everything in the working folder

In [ ]:
# Delete the working folder
!rm -rf /notebooks/*